# 00 — GridLock R2 — Full Run Guide

This notebook tells you **exactly what to run and in what order** to get from raw data → final demo output.

---

## Prerequisites installed?

Before running any notebook, verify your environment:

```bash
# From the project root (GridLock_R2_Transfer/)
venv\Scripts\activate
venv\Scripts\pip install shap   # Only needed for 06_shap.ipynb
```

---

## Execution Order

| Step | Notebook | What it does | Output |
|------|----------|--------------|--------|
| 1 | `01_eda.ipynb` | Explore raw violations CSV | EDA plots + `eda_summary.json` |
| 2 | `01b_features.ipynb` | Build features + encode categoricals | `features_encoded.parquet` + `label_encoders.pkl` |
| 3 | `02_cluster_tuning.ipynb` | Tune DBSCAN eps/min_samples on the full dataset | Silhouette score grid plots |
| 4 | `03_clustering.ipynb` | Run DBSCAN, build CIS table, aggregate grids | `zone_hour_grid.parquet`, `cis_table.parquet` |
| 5 | `04_training.ipynb` | Train 6 models, pick winner | `checkpoints/`, `eval_*.json` |
| 6 | `05_inference.ipynb` | Generate enforcement rankings + static HTML | `enforcement_priority_*.html` |
| 7 | `06_shap.ipynb` | SHAP analysis — feature importance + validation gate | `shap_summary.png`, `shap_pdp_hour.png` |

---

> **Can I skip steps?**  
> If you already ran notebooks 1–5 before and the processed files exist, you can jump
> directly to notebook 6 or 7. The cells below verify exactly which files are present.

## Cell 1 — Environment check
**Expected output**: All imports OK, Python 3.11+.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root : {PROJECT_ROOT}')
print(f'Python       : {sys.version.split()[0]}')

# Check key packages
packages = ['numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm', 'catboost',
            'loguru', 'tqdm', 'yaml', 'folium', 'shap']
missing = []
for pkg in packages:
    try:
        mod = __import__(pkg if pkg != 'sklearn' else 'sklearn')
        ver = getattr(mod, '__version__', '?')
        print(f'  ✓  {pkg:<15} {ver}')
    except ImportError:
        print(f'  ✗  {pkg:<15} NOT INSTALLED')
        missing.append(pkg)

if missing:
    print(f'\nInstall missing packages: venv\\Scripts\\pip install {" ".join(missing)}')
else:
    print('\nAll packages installed ✅')

Project root : C:\Users\USER\Desktop\GridLock_R2_Transfer
Python       : 3.12.10
  ✓  numpy           2.4.6
  ✓  pandas          2.2.3
  ✓  sklearn         1.9.0
  ✗  xgboost         NOT INSTALLED
  ✓  lightgbm        4.6.0
  ✗  catboost        NOT INSTALLED
  ✗  loguru          NOT INSTALLED
  ✓  tqdm            4.67.1
  ✓  yaml            6.0.2
  ✗  folium          NOT INSTALLED
  ✗  shap            NOT INSTALLED

Install missing packages: venv\Scripts\pip install xgboost catboost loguru folium shap


## Cell 2 — File state audit
**Expected output**: Green ✓ for all files that already exist, red ✗ for files that need to be generated.

This tells you **exactly which notebooks you need to run**.

In [2]:
import os

def check(path, label=None):
    p = Path(path)
    label = label or p.name
    if p.exists():
        size = p.stat().st_size / 1e3
        print(f'  ✓  {label:<50} {size:>8.1f} KB')
        return True
    else:
        print(f'  ✗  {label:<50}   MISSING')
        return False

R = PROJECT_ROOT
all_ok = True

print('=== RAW DATA ===')
all_ok &= check(R/'data'/'raw',                        'data/raw/  (directory)')

print('\n=== AFTER 01_eda.ipynb ===')
all_ok &= check(R/'data'/'outputs'/'eda_summary.json', 'data/outputs/eda_summary.json')

print('\n=== AFTER 01b_features.ipynb ===')
all_ok &= check(R/'data'/'processed'/'features_encoded.parquet',  'data/processed/features_encoded.parquet')
all_ok &= check(R/'data'/'processed'/'label_encoders.pkl',        'data/processed/label_encoders.pkl')

print('\n=== AFTER 03_clustering.ipynb ===')
all_ok &= check(R/'data'/'processed'/'features_with_zones.parquet', 'data/processed/features_with_zones.parquet')
all_ok &= check(R/'data'/'processed'/'zone_hour_grid.parquet',      'data/processed/zone_hour_grid.parquet')
all_ok &= check(R/'data'/'processed'/'zone_day_grid.parquet',       'data/processed/zone_day_grid.parquet')
all_ok &= check(R/'data'/'processed'/'cis_table.parquet',           'data/processed/cis_table.parquet')

print('\n=== AFTER 04_training.ipynb ===')
ckpt_dirs = sorted([d for d in (R/'checkpoints').glob('xgboost_hour_*') if d.is_dir()], reverse=True)
if ckpt_dirs:
    print(f'  ✓  {("checkpoints/" + ckpt_dirs[0].name):<50} (latest XGB checkpoint)')
else:
    print(f'  ✗  {"checkpoints/xgboost_hour_*":<50}   MISSING')
    all_ok = False

eval_files = list((R/'data'/'outputs').glob('eval_*.json'))
if eval_files:
    latest_eval = max(eval_files, key=lambda f: f.stat().st_mtime)
    print(f'  ✓  {("data/outputs/" + latest_eval.name):<50} {latest_eval.stat().st_size/1e3:.1f} KB')
else:
    print(f'  ✗  {"data/outputs/eval_*.json":<50}   MISSING')
    all_ok = False

print('\n=== AFTER 06_shap.ipynb ===')
check(R/'data'/'outputs'/'shap_summary.png',   'data/outputs/shap_summary.png')
check(R/'data'/'outputs'/'shap_importance.png','data/outputs/shap_importance.png')
check(R/'data'/'outputs'/'shap_pdp_hour.png',  'data/outputs/shap_pdp_hour.png')
check(R/'data'/'outputs'/'shap_report.json',   'data/outputs/shap_report.json')

print()
if all_ok:
    print('All critical files present ✅ — ready to run 06_shap.ipynb')
else:
    print('Some files missing ⚠️ — see ✗ rows above for which notebooks to run first')

=== RAW DATA ===
  ✓  data/raw/  (directory)                                  0.0 KB

=== AFTER 01_eda.ipynb ===
  ✗  data/outputs/eda_summary.json                        MISSING

=== AFTER 01b_features.ipynb ===
  ✗  data/processed/features_encoded.parquet              MISSING
  ✗  data/processed/label_encoders.pkl                    MISSING

=== AFTER 03_clustering.ipynb ===
  ✗  data/processed/features_with_zones.parquet           MISSING
  ✗  data/processed/zone_hour_grid.parquet                MISSING
  ✗  data/processed/zone_day_grid.parquet                 MISSING
  ✗  data/processed/cis_table.parquet                     MISSING

=== AFTER 04_training.ipynb ===
  ✗  checkpoints/xgboost_hour_*                           MISSING
  ✗  data/outputs/eval_*.json                             MISSING

=== AFTER 06_shap.ipynb ===
  ✗  data/outputs/shap_summary.png                        MISSING
  ✗  data/outputs/shap_importance.png                     MISSING
  ✗  data/outputs/shap_pdp_hou

## Cell 3 — How to open and run a notebook

**Option A — VS Code (recommended)**:
1. Open VS Code in the project root folder
2. Click on the notebook file in the Explorer panel
3. Click **"Select Kernel"** → choose **`venv (Python 3.11)`**
4. Click **"Run All"** (▶▶) at the top

**Option B — Jupyter in browser**:
```bash
# From project root
venv\Scripts\activate
venv\Scripts\jupyter notebook
# Browser opens at http://localhost:8888
# Navigate to notebooks/ and click the file
```

**Option C — Run from terminal (non-interactive)**:
```bash
venv\Scripts\jupyter nbconvert --to notebook --execute notebooks/06_shap.ipynb --output notebooks/06_shap_executed.ipynb
```

## Step-by-Step: Get Final Demo Output

Follow this exact sequence. Each step says what file to open and what final output you get.

---

### Step 1 — Run Feature Engineering (if not done)

**Only needed if `data/processed/features_encoded.parquet` is missing.**

Open: `notebooks/01b_features.ipynb` → Run All

✅ Done when you see:
```
Feature engineering complete.
features_encoded.parquet  saved  (XX MB)
label_encoders.pkl  saved
```

---

### Step 2 — Run Clustering (if not done)

**Only needed if `data/processed/zone_hour_grid.parquet` is missing.**

Open: `notebooks/03_clustering.ipynb` → Run All

✅ Done when you see:
```
CIS table saved → data/processed/cis_table.parquet  (N zones)
zone_hour_grid saved → data/processed/zone_hour_grid.parquet
zone_day_grid  saved → data/processed/zone_day_grid.parquet
```

---

### Step 3 — Run Training (Phase 1 retrain)

**Must be run after Step 2. This uses the new zone aggregate features.**

Open: `notebooks/04_training.ipynb` → Run All

Expected runtime: **5–10 minutes**

✅ Done when you see:
```
🏆 WINNER: xgboost_hour | NDCG@10=X.XXXX | MAE=X.XXXX
All results saved → data/outputs/eval_TIMESTAMP.json
```

---

### Step 4 — Generate Demo HTML (with time slider)

Open: `notebooks/05_inference.ipynb` → Run All

✅ Done when you see:
```
✓ Time-slider output saved → data/outputs/enforcement_slider_*.html
```
Open the `.html` file in Chrome — this is your demo.

---

### Step 5 — Run SHAP Analysis

Open: `notebooks/06_shap.ipynb` → Run All

Expected runtime: **2–4 minutes**

✅ Done when you see:
```
✅ SHAP validation gate PASSED
shap_summary.png saved
shap_pdp_hour.png saved
```

---

### Final outputs to use in the demo

| File | Use |
|------|-----|
| `data/outputs/enforcement_slider_DATE_Hh.html` | **Main demo** — open in Chrome, use time slider |
| `data/outputs/shap_summary.png` | Slide: "Our model is explainable" |
| `data/outputs/shap_pdp_hour.png` | Slide: "Hour-of-day effect on enforcement" |
| `data/outputs/shap_importance.png` | Slide: "Feature importance ranking" |
| `docs/demo_script.md` | Judge Q&A preparation |

## Cell 4 — Quick-launch: run the minimal path right now

If clustering and training are already done, this cell runs SHAP directly
and generates all plots in one go. Equivalent to running `06_shap.ipynb`.

**Expected runtime**: 2–4 minutes.

In [3]:
# ── Check prerequisites before launching ─────────────────────────────────────
prereqs = [
    PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet',
]
ckpt_dirs = sorted(
    [d for d in (PROJECT_ROOT / 'checkpoints').glob('xgboost_hour_*') if d.is_dir()],
    key=lambda d: d.name, reverse=True
)

missing = [f for f in prereqs if not f.exists()]
if missing:
    print('Cannot run — missing files:')
    for f in missing:
        print(f'  ✗ {f}')
    print('Run 03_clustering.ipynb then 04_training.ipynb first.')
elif not ckpt_dirs:
    print('Cannot run — no xgboost_hour_* checkpoint found. Run 04_training.ipynb first.')
else:
    print('Prerequisites OK. Launching 06_shap.ipynb via nbconvert...')
    print('(Plots will be saved to data/outputs/. Open 06_shap.ipynb to view inline.)')
    import subprocess
    result = subprocess.run(
        [
            sys.executable, '-m', 'jupyter', 'nbconvert',
            '--to', 'notebook',
            '--execute',
            '--ExecutePreprocessor.timeout=600',
            str(PROJECT_ROOT / 'notebooks' / '06_shap.ipynb'),
            '--output', str(PROJECT_ROOT / 'notebooks' / '06_shap_executed.ipynb'),
        ],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('\n✅ SHAP analysis complete!')
        out_dir = PROJECT_ROOT / 'data' / 'outputs'
        for fname in ['shap_summary.png', 'shap_importance.png', 'shap_pdp_hour.png', 'shap_report.json']:
            p = out_dir / fname
            if p.exists():
                print(f'  ✓  {fname}  ({p.stat().st_size / 1e3:.1f} KB)')
    else:
        print('❌ nbconvert failed. Open 06_shap.ipynb manually and run cells one by one.')
        print('stderr:', result.stderr[-2000:])

Cannot run — missing files:
  ✗ C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\zone_hour_grid.parquet
  ✗ C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\cis_table.parquet
Run 03_clustering.ipynb then 04_training.ipynb first.


## Cell 5 — Display SHAP plots (after 06_shap.ipynb is done)
Run this to view the saved plots inline in this notebook.

In [4]:
from IPython.display import Image, display

out_dir = PROJECT_ROOT / 'data' / 'outputs'
plots = [
    ('shap_summary.png',   'Global SHAP Summary (Beeswarm) — use in demo slides'),
    ('shap_importance.png','Feature Importance Bar Chart'),
    ('shap_pdp_hour.png',  'Hour-of-Day Effect — the ML vs baseline argument'),
    ('shap_pdp_rolling.png','rolling_7d_count — the strongest predictive signal'),
]

for fname, title in plots:
    p = out_dir / fname
    if p.exists():
        print(f'\n── {title} ──')
        display(Image(filename=str(p), width=800))
    else:
        print(f'\n{fname} not found — run 06_shap.ipynb first.')


shap_summary.png not found — run 06_shap.ipynb first.

shap_importance.png not found — run 06_shap.ipynb first.

shap_pdp_hour.png not found — run 06_shap.ipynb first.

shap_pdp_rolling.png not found — run 06_shap.ipynb first.


## Cell 6 — Print SHAP validation gate results
Quick check whether the Phase 1 fix worked correctly.

In [5]:
import json

report_path = PROJECT_ROOT / 'data' / 'outputs' / 'shap_report.json'
if not report_path.exists():
    print('shap_report.json not found. Run 06_shap.ipynb first.')
else:
    with open(report_path) as f:
        report = json.load(f)

    print('=== SHAP Report Summary ===')
    print(f'  Checkpoint       : {report["checkpoint"]}')
    print(f'  Sample size      : {report["sample_size"]:,} rows')
    print(f'  Expected value   : {report["expected_value"]:.4f}')
    print(f'  Top-5 features   : {report["top5_features"]}')
    print()

    gates = report['validation_gates']
    print('  Validation Gates:')
    print(f'    Gate 1 — zone_id absent:         {"✅ PASS" if gates.get("gate1_zone_id_absent") else "❌ FAIL"}')
    print(f'    Gate 2 — rolling in top-3:       {"✅ PASS" if gates.get("gate2_rolling_in_top3") else "⚠️  WARN"}')
    print(f'    Gate 3 — hour_of_day in top-5:   {"✅ PASS" if gates.get("gate3_hour_in_top5") else "⚠️  WARN"}')
    print()
    overall = report.get('overall_gate_passed', False)
    print(f'  Overall: {"✅ PASSED — model is well-behaved" if overall else "❌ FAILED — review Phase 1"}')

    print()
    print('  Top-10 Feature Importance:')
    for i, (feat, val) in enumerate(report['top10_mean_abs_shap'].items(), start=1):
        print(f'    {i:2d}. {feat:<35} {val:.4f}')

shap_report.json not found. Run 06_shap.ipynb first.
